# 📘 Guide 09: AI APIs, costs and Ollama

**Who it's for:** you'd like to know what a lab will cost before you run it, or you'd rather run models for free on your own machine. **Time:** about 25 minutes.

Good news first: with the default `MODEL_MODE=cheap` (OpenAI's `gpt-6-luna`, through OpenRouter) a typical lab call costs a tenth of a cent or less, and the whole course costs a few dollars at most. `MODEL_MODE=free` uses OpenRouter's free models for $0, within a daily request limit. One key, `OPENROUTER_API_KEY`, covers both. This guide shows you where the money goes and how to keep it low, or at zero.

### 🎯 You'll learn
1. Tokens: the unit everything is priced in
2. Estimating the cost of grading a whole class
3. `MODEL_MODE` (cheap / free / custom) and `DATA_SIZE` in `.env`
4. Running free local models with Ollama
5. Swapping providers without changing code

## Part 1: Tokens

Models don't read words; they read **tokens**, chunks of roughly 4 characters of English (about ¾ of a word). You pay for:
- **input tokens**: your prompt, the rubric, retrieved course notes, the student's answer
- **output tokens**: what the model writes back (usually priced 4–8× higher)

A quick estimate is good enough for budgeting:

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough rule of thumb: ~4 characters per token for English."""
    return max(1, round(len(text) / 4))


rubric = "Award 2 marks for defining a stack as LIFO, 2 for push/pop, 1 for a real-world example. " * 3
answer = "A stack is a data structure where the last element pushed is the first popped, like plates. " * 4
print(f"Rubric ≈ {estimate_tokens(rubric)} tokens, answer ≈ {estimate_tokens(answer)} tokens")

try:
    import tiktoken  # an exact tokenizer for OpenAI models, if installed

    enc = tiktoken.get_encoding("o200k_base")
    print(f"tiktoken says the answer is exactly {len(enc.encode(answer))} tokens")
except Exception:
    print("(tiktoken not available offline: the estimate is fine for budgeting)")

## Part 2: What does grading a class cost?

Prices are quoted **per million tokens** and change often, so check your provider's pricing page. The numbers below were checked in November 2026: `gpt-6-luna` is the course default, `gpt-6.1-terra` an optional stronger grader (OpenRouter passes both prices through, plus a small fee when you buy credits), and OpenRouter's free models cost nothing.

In [ ]:
# 💡 USD per 1M tokens, checked November 2026. ✋ Prices change: confirm them on the pricing pages.
PRICES = {
    "gpt-6-luna (cheap)": {"input": 0.10, "output": 0.50},
    "gpt-6.1-terra (custom)": {"input": 1.00, "output": 5.00},
    "openrouter/free (free)": {"input": 0.00, "output": 0.00},
}

students, questions = 30, 5
input_per_call = 2_500  # rubric + retrieved notes + answer + instructions
output_per_call = 400  # marks, evidence and feedback (⚠️ reasoning models bill their hidden "thinking" here too)

calls = students * questions
for name, p in PRICES.items():
    cost = calls * (input_per_call * p["input"] + output_per_call * p["output"]) / 1_000_000
    print(f"{name:>22}: {calls} calls → ${cost:.2f} for Emma's whole CS201 class")

That's why the course's default is one cheap model for everything. A stronger grader is an optional upgrade, not a requirement.

## Part 3: `MODEL_MODE` and `DATA_SIZE` in `.env`

GradeWise never hard-codes a model name. Two lines at the top of `.env` decide what a run uses:

```
MODEL_MODE=cheap     # cheap (default) | free | custom
DATA_SIZE=full       # full (default) | small
```

| `MODEL_MODE=` | Chat, grading and vision | Embeddings | Key you need | Cost |
|---|---|---|---|---|
| `cheap` | `openrouter:openai/gpt-6-luna` (grading uses a higher reasoning effort) | `openrouter:openai/text-embedding-3-small` | `OPENROUTER_API_KEY` (from openrouter.ai/keys), with a few dollars of credits | ~$0.10 / $0.50 per 1M tokens in / out |
| `free` | `openrouter:openrouter/free` (picks a free model per request) | `openrouter:nvidia/nemotron-3.5-embed-1b:free` | the same `OPENROUTER_API_KEY`, no credits | $0 |
| `custom` | whatever `CHAT_MODEL`, `SMART_MODEL` and `VISION_MODEL` say, e.g. `openai:gpt-6-luna`, `google_genai:...`, `anthropic:...`, `ollama:llama3.2` | `EMBEDDING_MODEL` | whichever provider you pick (`OPENAI_API_KEY`, `GOOGLE_API_KEY`, `ANTHROPIC_API_KEY`, none for Ollama) | yours |

- `cheap` and `free` **ignore** the four `*_MODEL` lines, so switching is a one-line change (restart the kernel afterwards).
- **Free mode caveats:** (1) **rate limits** of 20 requests a minute and **50 a day** (1,000 a day once you've ever bought $10 of credits); GradeWise retries "429" errors with a back-off and runs at most 2 calls at once. (2) So **use it with `DATA_SIZE=small`**: a full class run makes hundreds of calls. (3) **The model varies per call**, so results vary and an occasional structured-output failure is normal (GradeWise retries); and free providers may log or train on your prompts: fine for this course's made-up data, never for real student data. You may need to allow "free endpoints that may train on inputs" in your OpenRouter privacy settings.
- **Optional upgrade:** `MODEL_MODE=custom` with `SMART_MODEL=openrouter:openai/gpt-6.1-terra` ($1 / $5 per 1M tokens) gives you a stronger grader, at ten times the price.
- **`DATA_SIZE=small`** is for when you know the ideas and just want to run the APIs quickly: 2 PDFs (Unit 1 and the handbook), questions Q1–Q3 and 2 students, instead of 4 PDFs, all 10 questions and 10 students.

Let's see the logic from `gradewise/config.py` and `gradewise/datasets.py` (this just reads settings; it makes no API calls):

In [ ]:
from gradewise import datasets
from gradewise.config import Settings

for mode in ("cheap", "free", "custom"):
    s = Settings(model_mode=mode)
    keys = ", ".join(s.required_keys) or "none"
    print(f"MODEL_MODE={mode:<6} → grading uses {s.grading_model:<36} key needed: {keys}")

print()
for small in (False, True):
    print(datasets.describe(small=small))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Protect your wallet</h2>
            <span style="color:#22c55e;">Set a <b>monthly spending limit</b> on your provider's billing page before you start. It's the single best protection against a runaway loop. For a course like this, $5–10 is plenty.</span>
        </td>
    </tr>
</table>

## Part 4: Free local models with Ollama

[Ollama](https://ollama.com) runs open models on your own computer: no key, no cost, and your students' data never leaves the machine (a real plus for schools).

Install it:
- **Windows / Mac:** download the installer from ollama.com
- **Linux:** `curl -fsSL https://ollama.com/install.sh | sh`

Then, in a terminal (same on every OS):
```bash
ollama pull llama3.2          # download a small model (~2 GB)
ollama run llama3.2           # chat with it in the terminal; /bye to quit
ollama list                   # what you have installed
```

And point GradeWise at it in `.env` with the `custom` mode:
```
MODEL_MODE=custom
CHAT_MODEL=ollama:llama3.2
SMART_MODEL=ollama:llama3.2
VISION_MODEL=ollama:llama3.2
```

Local models are slower and less capable than frontier ones, especially at reading handwriting. They're great for following along with weeks 1–3 for free, and your data never leaves your machine. (The easiest $0 route, though, is `MODEL_MODE=free`: no install, just your OpenRouter key.)

In [ ]:
import httpx

# 💡 Is Ollama running? This only talks to your own machine, and it's fine if it isn't installed.
try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=1.0)
    models = [m["name"] for m in r.json().get("models", [])]
    print("✅ Ollama is running. Models:", models or "none pulled yet")
except Exception:
    print("ℹ️ Ollama isn't running on this machine (that's fine). Install it from ollama.com to use free local models.")

## Part 5: Same code, any provider

Because labs use LangChain's `init_chat_model`, switching provider is a `.env` change. In code it looks like this (shown, not run, since it needs a key or Ollama):

```python
from langchain.chat_models import init_chat_model

llm = init_chat_model("openrouter:openai/gpt-6-luna")    # or direct: "openai:gpt-6-luna", "google_genai:...", "ollama:llama3.2"
reply = llm.invoke("In one sentence, what is a stack?")
print(reply.text)                   # 💡 .text: the reply's text, even when content is a list of blocks
print(reply.usage_metadata)   # 💡 input_tokens, output_tokens: the numbers you pay for
```

See `setup/SWAP_PROVIDER.md` for every supported provider.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Check <code>PRICES</code> in Part 2 against today's real prices for the models your <code>MODEL_MODE</code> uses, and work out the cost of grading 120 students × 8 questions. Then, if you can, install Ollama, pull a small model and re-run the check cell.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• OpenAI pricing: <a href="https://developers.openai.com/api/docs/pricing">developers.openai.com/api/docs/pricing</a><br/>• OpenRouter models and prices: <a href="https://openrouter.ai/models">openrouter.ai/models</a><br/>• OpenRouter API keys: <a href="https://openrouter.ai/keys">openrouter.ai/keys</a><br/>• OpenAI tokenizer playground: <a href="https://platform.openai.com/tokenizer">platform.openai.com/tokenizer</a><br/>• Ollama: <a href="https://ollama.com">ollama.com</a><br/>• LangChain init_chat_model: <a href="https://docs.langchain.com/oss/python/langchain/models">docs.langchain.com/oss/python/langchain/models</a></span>
        </td>
    </tr>
</table>